# STAT 764 · Meeting 14 — A range you can bet on: conformal prediction

**Thursday, October 22**

Every model so far has given one number per case: a price, a probability. Mini-project 1's sealed
envelope asks for more — an AUC **with an interval**, a range for one number you have not seen yet.
Today does the same for a single prediction: a range for one house's sale price, from **any**
model, with a promise you can check.

**The question for today.** A model says a house will sell for about \$136,000. How far off could
that be — and can we promise anything without assuming anything about the model or its errors?

Today has two halves: conformal prediction, with a short studio, and then the rest of class for
your Mini-project 1 team.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stat764 import load

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Ames house prices, with the columns from Meetings 5-8.
ames = load("ames.csv")
NUM = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF", "Lot_Area"]
CAT = ["Neighborhood", "Central_Air"]
X, y = ames[NUM + CAT], ames["SalePrice"].to_numpy()


def make_pipe(model):
    """Meetings 5-8's preprocessing, with any model at the end."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="Missing")),
                              ("encode", OneHotEncoder(handle_unknown="ignore"))]), CAT),
        ])),
        ("model", model),
    ])


# Three groups of houses, chosen at random: half to FIT the model, a quarter to CALIBRATE
# the range, and a quarter to TEST whether the promise holds. No house is in two groups.
rows = np.arange(len(y))
fit_rows, rest = train_test_split(rows, test_size=0.5, random_state=764)
cal_rows, test_rows = train_test_split(rest, test_size=0.5, random_state=764)
print(f"fit {len(fit_rows):,} houses · calibrate {len(cal_rows):,} · test {len(test_rows):,}")

## 1. One house, one number

In [ ]:
# The plain linear model from Meetings 5-6, fitted on the fit houses only.
model = make_pipe(LinearRegression()).fit(X.iloc[fit_rows], y[fit_rows])

# One test house: a house the model has never seen.
house = test_rows[0]
guess = model.predict(X.iloc[[house]])[0]
print(f"predicted ${guess:,.0f} · actual ${y[house]:,.0f} · off by ${abs(y[house] - guess):,.0f}")

Off by about \$14,000 this time. The next house could be off by much more, or much less. What we
want is a **range** that contains the true price for 90% of new houses — and a reason to believe
it that does not depend on the model being right.

## 2. Split conformal prediction, in five steps

1. **Split** the houses: fit, calibrate, test. *(Done above.)*
2. **Fit** any model on the fit houses. *(Done: `model`.)*
3. **Measure the errors on the calibration houses** — houses the model never saw:
   |actual − predicted| for each.
4. **Line the errors up from smallest to largest and take the k-th,** where
   k = ⌈(n + 1) × 0.90⌉. With n = 732 calibration houses, k = ⌈733 × 0.90⌉ = 660: the **660th
   smallest** error. Call it **q**.
5. **For any new house: prediction ± q.**

**Two jobs, two sets of houses.** The model's job is to predict well; the fit houses train it. The
calibration houses' job is to say how wide the range must be. A better model makes smaller
errors, so its range is narrower — but even a poor model gets a range that keeps the promise.

In [ ]:
level = 0.90

# Step 3: how far off is the model on houses it never saw? Five of the 732, to see what an error is.
pred_cal = model.predict(X.iloc[cal_rows])
errors = np.abs(y[cal_rows] - pred_cal)
print(pd.DataFrame({"actual price": y[cal_rows], "predicted": pred_cal, "error": errors})
      .head().map(lambda v: f"${v:,.0f}").to_string(index=False))

In [ ]:
# Step 4: line the 732 errors up from smallest to largest, and take the k-th.
n = len(errors)
k = int(np.ceil((n + 1) * level))                 # ⌈733 × 0.90⌉ = 660
q = np.sort(errors)[k - 1]                        # the 660th smallest ([k - 1]: Python counts from 0)
print(f"n = {n} calibration errors · k = ⌈({n} + 1) × {level}⌉ = {k} · q = the {k}th smallest = ${q:,.0f}")

# Step 5: the prediction, plus or minus q.
print(f"house {house}: ${guess:,.0f} ± ${q:,.0f} = ${guess - q:,.0f} to ${guess + q:,.0f}"
      f"   (actual ${y[house]:,.0f})")

# The 732 errors, and where q cuts them. (A few errors over $150,000 are stacked at the right edge.)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(np.minimum(errors, 150_000) / 1000, bins=30, range=(0, 150), color="#9ab")
ax.axvline(q / 1000, color="k", lw=2)
ax.set_xlabel(f"calibration error, $ thousands ({np.sum(errors > 150_000)} over 150 shown at 150)")
ax.set_ylabel("houses")
ax.set_title(f"{k} of the {n} errors are at or left of the line (q = ${q / 1000:,.1f}k)")
plt.show()

**Why the promise holds — a line-up of 733 errors.** Now bring in a new house, drawn the same way
as the calibration houses: at random, from the same houses. Put its error in the line-up with
the 732 calibration errors — 733 errors in all. Nothing distinguishes the new one from the others,
so it is **equally likely to land in any of the 733 places**. Its range catches its price exactly
when its error is no bigger than q, the 660th smallest calibration error — that is, when it lands
in one of the first 660 places. The chance:

> 660 / 733 = 90.04%.

That is the whole argument: no bell curve, nothing about the model. It works for a linear
regression, a random forest, or anything else that makes a prediction. And the "+ 1" in
k = ⌈(n + 1) × 0.90⌉ is the new house taking its place in the line-up.

## 3. Check the promise on the test houses

In [ ]:
# Every test house gets its own range: its prediction plus or minus the same q.
pred_test = model.predict(X.iloc[test_rows])
inside = np.abs(y[test_rows] - pred_test) <= q
print(f"{inside.mean():.1%} of {len(test_rows)} test houses are inside their range (promised: 90%)")

**Promised 90%, got less. Is the promise broken?** Write your guess before you run the next cell.

The promise is about the **method**: over many random splits, the range catches at least 90% of
new houses on average. Any one set of 733 test houses can land a little above or below. To see
how much, wrap the five steps in a function and run them on 200 different random splits.

In [ ]:
def split_conformal(pipe, fit_rows, cal_rows, test_rows, level=0.90, X=X, y=y):
    """The five steps in one function. Returns q (the ± margin) and the share of test cases
    whose true value lands inside prediction ± q."""
    pipe.fit(X.iloc[fit_rows], y[fit_rows])                                   # step 2
    errors = np.abs(y[cal_rows] - pipe.predict(X.iloc[cal_rows]))             # step 3
    n = len(errors)
    k = int(np.ceil((n + 1) * level))
    q = np.sort(errors)[k - 1]                                                # step 4
    inside = np.abs(y[test_rows] - pipe.predict(X.iloc[test_rows])) <= q      # step 5, checked
    return q, inside.mean()


# The same recipe on 200 other random splits of the same houses (about 10 seconds).
covered = []
for seed in range(200):
    f, r = train_test_split(rows, test_size=0.5, random_state=seed)
    c, t = train_test_split(r, test_size=0.5, random_state=seed)
    covered.append(split_conformal(make_pipe(LinearRegression()), f, c, t)[1])
covered = np.array(covered)
print(f"200 splits: average {covered.mean():.1%} · middle 95% from "
      f"{np.percentile(covered, 2.5):.1%} to {np.percentile(covered, 97.5):.1%}")
print(f"our split, {inside.mean():.1%}, is below {np.mean(covered > inside.mean()):.0%} of them")

**What this shows.** On average the range catches 90% — the promise holds. Our split was simply
an unlucky one. Any single set of 733 houses wobbles by a few points either way, which is the same
lesson as Mini-project 1's sealed envelope: an AUC computed on one set of 4,119 calls moves too.

## 4. Choose the level — and what 90% does not promise

In [ ]:
# The same model and split, at four levels. A stronger promise needs a wider range.
for lv in (0.80, 0.90, 0.95, 0.99):
    q_lv, cov = split_conformal(make_pipe(LinearRegression()), fit_rows, cal_rows, test_rows, lv)
    print(f"  {lv:.0%}: ± ${q_lv:>9,.0f}   covers {cov:.1%} of the test houses")

### The promise, stated exactly

**The promise.** For a new house, split conformal prediction at level 90% guarantees

> P( the house's actual price lies inside its prediction ± q ) ≥ 0.90.

With our 732 calibration houses the chance is exactly 660/733 = 90.04% when no two errors tie, and
never more than 0.90 + 1/(n + 1): the range is not padded. It holds exactly for these 732 houses,
not only in large samples.

**What it needs — and nothing else.**

1. **Exchangeable houses.** The calibration houses and the new house must be *exchangeable*:
   every ordering of them is equally likely, so the order of the rows carries no information
   (Meeting 6's word). Houses drawn independently from the same population — which is what a
   random split of one dataset gives you — are exchangeable. The reading states it this way, as
   independent and identically distributed.
2. **The model never saw the calibration houses.** It was fitted on other houses, so its errors on
   the calibration houses are honest — Meeting 12's rule again. Measure the errors on the houses it
   was fitted on and they come out too small.
3. **One recipe for every house:** the same fitted model, the same error |actual − predicted|, the
   same q.

**Not needed:** a correct model, a straight line, bell-shaped errors, errors of equal size for
every house, or a large sample. A poor model still keeps the promise — with a wide range.
Conformal prediction does not make a model better; it makes the model honest about how good it is.

**The sense in which it holds.**

- **On average over the luck of the draw** — over which houses landed in the calibration set
  *and* which new house comes along. It is a promise about the recipe, used again and again:
  Section 3's 200 splits averaged 90.0% — the 660/733 the line-up predicts.
- **Not for one particular calibration set.** Once the 732 houses are fixed, the share of future
  houses caught lands a little above or below 90% — 87.3% of our test houses. More calibration
  houses means less wobble.
- **Not for every kind of house.** 90% is an average over all houses together: *marginal*
  coverage. It can be far higher for cheap houses and far lower for expensive ones *(studio
  question 3)*. A promise for every kind of house — *conditional* coverage — cannot be guaranteed
  in general without assuming more about the data.
- **Not for a different population.** When new houses are drawn differently from the calibration
  houses — a later year, another city — requirement 1 fails and the promise is off *(studio
  question 2)*.

## Studio

About twenty minutes: two questions, the third if you have time. **Then the rest of class is for
your Mini-project 1 team.**

The tool is `split_conformal(pipe, fit_rows, cal_rows, test_rows, level)`, which returns
`(q, coverage)`: the ± margin, and the share of test cases inside their range. It takes `X=` and
`y=` for a different dataset.

**Before each question, write your guess** as a comment in its cell. Keep it when the answer
disagrees — the Compare uses it. The notebook is due 24 hours after class.

### 1. Any model

- **(a)** Run `split_conformal` at 90% with a random forest in place of the linear model:
  `make_pipe(RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0))`, on the
  same `fit_rows`, `cal_rows` and `test_rows`. Print q and the coverage.
- **(b)** Put them beside the linear model's (Section 4's 90% line).

**Answer in a sentence:** does the forest keep the promise — and which range would you rather hand
a buyer, and why?

### 2. Break it: next year's riders

The bike-share data from Meeting 6: hourly rentals in Washington, D.C., in 2011 and 2012. Run the
setup cell below first; it makes `Xb`, `yb`, `bike_pipe`, and the hour numbers `rows_2011`,
`rows_2012` and `all_rows`.

- **(a)** **2011 only.** Split `rows_2011` at random, 70% to fit and 30% to calibrate
  (`train_test_split(rows_2011, test_size=0.3, random_state=764)`), and test on **all** of
  `rows_2012`. Run `split_conformal(bike_pipe(LinearRegression()), ..., X=Xb, y=yb)` at 90%.
- **(b)** The same with `bike_pipe(RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
  random_state=0))`.
- **(c)** **For contrast:** both models again on a random split of `all_rows` — half to fit, a
  quarter to calibrate, a quarter to test, as in the setup cell for the houses.

**Answer in a sentence:** why does the promise fail on 2012 — and why does the model with the
narrower range fail worse?

### 3. Every kind of house? *(if you have time)*

- **(a)** Use Section 3's `inside` (inside the range or not, for each test house) and `pred_test`
  (its predicted price).
- **(b)** Split the test houses into thirds by predicted price,
  `pd.qcut(pred_test, 3, labels=["cheapest third", "middle third", "priciest third"])`, and print
  the coverage in each third: `pd.Series(inside).groupby(thirds).mean()`.

**Answer in a sentence:** does the 90% hold for the most expensive houses — and what would you tell
someone buying one?

In [ ]:
# Studio setup for question 2 -- run this before you start it.
bike = load("bike_hour.csv")
FEAT = [c for c in bike.columns
        if c not in ("cnt", "casual", "registered", "dteday", "instant", "yr")]
Xb, yb = bike[FEAT], bike["cnt"].to_numpy()
rows_2011 = np.where(bike["yr"] == 0)[0]                    # every hour of 2011
rows_2012 = np.where(bike["yr"] == 1)[0]                    # every hour of 2012
all_rows = np.arange(len(yb))


def bike_pipe(model):
    """Meeting 6's bike pipeline: fill, scale, any model."""
    return Pipeline([("fill", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("model", model)])


print(f"riders per hour: 2011 {yb[rows_2011].mean():.0f} on average · 2012 {yb[rows_2012].mean():.0f}")

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of next class

We compare answers at the start of next class, built from the notebooks you upload. So leave
these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 1: the forest's q and coverage beside the linear model's.
3. Question 2: the coverage on 2012 for both models, and on the random split.

## Exit ticket

> A 90% conformal range for one house is \$90,000 to \$182,000. In two or three sentences: what
> does the 90% promise — and name one situation where it would not hold.

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio
notebook — Day 14". Your two studio points need this answered: a blank exit ticket earns zero.
It is not graded for being right — it is how I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **Angelopoulos & Bates (2021), "A Gentle Introduction to Conformal Prediction and
Distribution-Free Uncertainty Quantification,"** arXiv 2107.07511 — this week's reading. Section 1,
about three pages, is the same recipe shown for sorting images rather than pricing houses, with the
promise stated as its Theorem 1.

📗 **MAPIE** — the scikit-learn-compatible library that wraps any model and returns conformal
ranges in one call.

📗 **Conformalized quantile regression** (Romano, Patterson & Candès, 2019) — ranges that widen
where the model is less sure, which is the repair for studio question 3.

📗 **The jackknife+ and CV+** (Barber, Candès, Ramdas & Tibshirani, 2021) — the same promise without
setting a quarter of the data aside to calibrate.

🎓 **Meeting 26** comes back to "90% on average, but not for every group" — models that work for
some people and not others.